
# QSO continuum with Lyman-alpha forest absorption at z=3

A power-law QSO continuum (Vanden Berk et al. 2001 composite slope
$\alpha_{\nu} = -0.5$) is built with tengri's AGN multicolor disc
and pure-stellar synthesis. The Inoue et al. 2014 intergalactic-medium
transmission is then applied to the observed frame, suppressing the
blue side below $\lambda_{\rm obs} < \lambda_{\rm Ly\alpha}(1+z)$.

The Lyman valley (the deep depression between Lyman-α at 1216 Å and
Lyman-β at 1026 Å in the rest frame) sweeps through the observer's
window, accumulating dark matter overdensity absorption. At z = 3,
:math:`\lambda_{\rm Ly\alpha}^{\rm obs} = 4864$ Å (deep optical).

Key observables:
- **Intrinsic QSO slope**: visible above Lyman limit (912 Å rest)
- **Lyman forest**: mottled forest shortward of Lyα (4864 Å observed at z=3)
- **Lyman valley**: sharp darkening between Lyα and Lyβ limits

## References
.. [1] Vanden Berk et al. 2001, AJ, 122, 549
       *Composite Quasar Spectra from the Sloan Digital Sky Survey*
.. [2] Inoue et al. 2014, MNRAS, 442, 1805
       *Observational Constraints on Cosmic Reionization*


In [ ]:
import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")

# ============================================================================
# Constants
# ============================================================================

C_AA_PER_S = 2.998e18  # Speed of light in Angstrom / second
Z_QSO = 3.0

# Lyman series wavelengths in vacuum (rest frame)
# Source: NIST Atomic Spectra Database
LY_ALPHA_REST = 1215.67  # Angstrom
LY_BETA_REST = 1025.72
LY_LIMIT_REST = 912.0

# ============================================================================
# Build QSO SED model
# ============================================================================

ssp = tengri.load_ssp()

model = tengri.SEDModel.build(
    ssp,
    # Pure stellar synthesis: single-age stellar pop for normalization only
    sfh={"type": "dpl", "*": tengri.FIXED, "tau_gyr": 1.0, "log_peak_sfr": 1.0, "alpha": 1.5, "beta": 1.0},
    dust={"type": "two_component", "*": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0},
    agn={
        "type": "composable",
        "disc": {"type": "multicolor", "*": tengri.FIXED},  # Vanden Berk slope α_ν = -0.5 (Kubota & Done 2018)
        "*": tengri.FIXED,
        "log_lbol": 12.5,  # L_bol ~ 3e12 L_sun (typical luminous QSO)
        "frac": 1.0,  # 100% AGN dominance
    },
    # Lyman-alpha forest absorption via IGM transmission
    igm={"type": "inoue14", "*": tengri.FIXED},
    redshift=tengri.Fixed(Z_QSO),
)

# Sample default parameters
params = dict(model.spec.sample(jax.random.PRNGKey(0)))

# Predict rest-frame SED
out = model.predict_rest_sed(params)
wave_rest = np.asarray(out.wavelength)  # Rest-frame Angstrom
sed_rest = np.asarray(out.sed)          # Rest-frame erg/s/Hz

# Transform to observer frame
wave_obs = wave_rest * (1.0 + Z_QSO)
nu_obs = C_AA_PER_S / wave_obs
nu_L_nu = nu_obs * sed_rest

# ============================================================================
# Synthesize intrinsic (no IGM) SED for reference
# ============================================================================

# Create a second model without IGM to show intrinsic continuum
model_no_igm = tengri.SEDModel.build(
    ssp,
    sfh={"type": "dpl", "*": tengri.FIXED, "tau_gyr": 1.0, "log_peak_sfr": 1.0, "alpha": 1.5, "beta": 1.0},
    dust={"type": "two_component", "*": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0},
    agn={
        "type": "composable",
        "disc": {"type": "multicolor", "*": tengri.FIXED},
        "*": tengri.FIXED,
        "log_lbol": 12.5,
        "frac": 1.0,
    },
    # No IGM transmission
    redshift=tengri.Fixed(Z_QSO),
)

out_no_igm = model_no_igm.predict_rest_sed(params)
sed_rest_no_igm = np.asarray(out_no_igm.sed)
nu_L_nu_no_igm = nu_obs * sed_rest_no_igm

# ============================================================================
# Plot
# ============================================================================

fig, ax = plt.subplots(figsize=(8.0, 5.0))

# Intrinsic QSO power-law (no IGM)
ax.loglog(
    wave_obs,
    nu_L_nu_no_igm,
    color="C0",
    lw=2.0,
    label=r"QSO intrinsic (no IGM)",
    alpha=0.8,
)

# With Lyman-alpha forest absorption (Inoue+2014)
ax.loglog(
    wave_obs,
    nu_L_nu,
    color="C3",
    lw=2.0,
    label=r"QSO with Lyman-$\alpha$ forest (Inoue+2014)",
    alpha=0.9,
)

# ============================================================================
# Mark Lyman series limits in observed frame
# ============================================================================

colors_lyman = {"alpha": "#d62728", "beta": "#ff7f0e", "limit": "#2ca02c"}

# Lyman-alpha limit in observed frame
ly_alpha_obs = LY_ALPHA_REST * (1.0 + Z_QSO)
ax.axvline(ly_alpha_obs, color=colors_lyman["alpha"], lw=1.0, ls="--", alpha=0.6)
ax.text(ly_alpha_obs * 0.95, ax.get_ylim()[1] * 0.3, r"$\lambda_{\rm Ly\alpha}^{\rm obs}$",
        fontsize=9, color=colors_lyman["alpha"], ha="right", rotation=90, va="bottom")

# Lyman-beta limit in observed frame
ly_beta_obs = LY_BETA_REST * (1.0 + Z_QSO)
ax.axvline(ly_beta_obs, color=colors_lyman["beta"], lw=1.0, ls="--", alpha=0.6)
ax.text(ly_beta_obs * 0.95, ax.get_ylim()[1] * 0.15, r"$\lambda_{\rm Ly\beta}^{\rm obs}$",
        fontsize=9, color=colors_lyman["beta"], ha="right", rotation=90, va="bottom")

# Lyman limit (912 Å) in observed frame
ly_limit_obs = LY_LIMIT_REST * (1.0 + Z_QSO)
ax.axvline(ly_limit_obs, color=colors_lyman["limit"], lw=1.2, ls=":", alpha=0.7)
ax.text(ly_limit_obs * 0.95, ax.get_ylim()[1] * 0.05, r"Lyman limit",
        fontsize=8, color=colors_lyman["limit"], ha="right", rotation=90, va="bottom")

# Shade the Lyman valley (absorption bottleneck between Lyβ and Lyα)
ax.axvspan(ly_beta_obs, ly_alpha_obs, color="gray", alpha=0.08, zorder=0)
ax.text((ly_beta_obs + ly_alpha_obs) / 2, ax.get_ylim()[1] * 0.7,
        "Lyman valley\n(dark forest)",
        fontsize=9, color="0.5", ha="center", va="center")

# ============================================================================
# Labels and limits
# ============================================================================

ax.set_xlim(800, 5e4)
ax.set_ylim(1e38, 1e44)

ax.set_xlabel(r"Observed wavelength $\lambda_{\rm obs}$ [$\mathrm{\AA}$]", fontsize=11)
ax.set_ylabel(r"$\nu L_{\nu}$  [erg s$^{-1}$]", fontsize=11)
ax.set_title(
    rf"QSO at $z = {Z_QSO}$ with Lyman-$\alpha$ Forest Absorption",
    fontsize=12,
    fontweight="normal",
)

ax.legend(frameon=False, fontsize=10, loc="lower left")
ax.grid(True, alpha=0.2, which="both")

fig.tight_layout()
plt.savefig("plot_lyman_alpha_forest_QSO_template.png", dpi=150, bbox_inches="tight")